# Preprocessing

In [1]:
import sys
from pathlib import Path
repo_root = Path().resolve().parent  
sys.path.append(str(repo_root))

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from src.data.Tenth_Best_Time_Estimator import TenthBestTimeEstimator

In [3]:
raw_parquet = pd.read_parquet('../data/raw/reunion_segments.parquet')
df_parquet = pd.DataFrame(raw_parquet)

In [4]:
tbte_ride = TenthBestTimeEstimator(repo_root / "src" / "models" / "missing_tenth_best_time_estimator_ride.pkl")
tbte_run = TenthBestTimeEstimator(repo_root / "src" / "models" / "missing_tenth_best_time_estimator_run.pkl")

df_parquet_missing_tbt = df_parquet[df_parquet['tenth_best_time'].isna()]
df_parquet_missing_tbt_ride = df_parquet_missing_tbt[df_parquet_missing_tbt['activity_type'] == 'Ride'].copy()
df_parquet_missing_tbt_run = df_parquet_missing_tbt[df_parquet_missing_tbt['activity_type'] == 'Run'].copy()

df_parquet_adding_tbt_ride = tbte_ride.add_tenth_best_time(df_parquet_missing_tbt_ride)
df_parquet_adding_tbt_run = tbte_run.add_tenth_best_time(df_parquet_missing_tbt_run)

df_existing = df_parquet.dropna(subset=['tenth_best_time'])

df_parquet_clean = pd.concat([df_existing, df_parquet_adding_tbt_ride, df_parquet_adding_tbt_run], ignore_index=True)
df_parquet_clean = df_parquet_clean.drop(columns=["log_efforts", "log_athletes"])

if 'id' in df_parquet_clean.columns:
    # rename the columns to match the expected output
    df_parquet_clean = df_parquet_clean.rename(columns={"id": "segment_id"})

In [5]:
# save_to_parquet
output_path = repo_root / "data" / "processed" / "reunion_segments_cleaned.parquet"
df_parquet_clean.to_parquet(output_path, index=False)

## Delete lines

In [ ]:
### delete lines when "altitude_profile" is missing 
df_parquet = df_parquet.dropna(subset=['altitude_profile'])

### save the cleaned dataframe
#df_parquet.to_parquet('../data/processed/reunion_segments_cleaned.parquet', index=False)

In [7]:
df_parquet_cleaned = pd.read_parquet('../data/processed/reunion_segments_cleaned.parquet')

## Missing tenth best time

### Model

In [5]:
df_clean = df_parquet.dropna(subset=['tenth_best_time'])

X = df_clean[['average_top_10_time', 'best_time', 'total_effort_count', 'total_athlete_count']].copy()
X['log_efforts'] = np.log(df_clean['total_effort_count'] + 1)
X['log_athletes'] = np.log(df_clean['total_athlete_count'] + 1)
X = X[['average_top_10_time', 'best_time', 'log_efforts', 'log_athletes']]

y = df_clean['tenth_best_time']

In [6]:
X_ride = X[df_clean['activity_type'] == 'Ride']
y_ride = y[df_clean['activity_type'] == 'Ride']
X_run = X[df_clean['activity_type'] == 'Run']
y_run = y[df_clean['activity_type'] == 'Run']

In [9]:
X_ride_train, X_ride_test, y_ride_train, y_ride_test = train_test_split(X_ride, y_ride, test_size=0.2, random_state=42)

model_ride = LinearRegression()
model_ride.fit(X_ride_train, y_ride_train)
print("Coefficients:", ", ".join(f"{coef:.2f}" for coef in model_ride.coef_))
print(f"Intercept: {model_ride.intercept_:.2f}")

print(f"Ride Model R^2 on test set: {model_ride.score(X_ride_test, y_ride_test): .2f}")

# save model
import joblib
joblib.dump(model_ride, '../src/models/missing_tenth_best_time_estimator_ride.pkl')

Coefficients: 1.82, -0.81, 16.95, -25.35
Intercept: 19.26
Ride Model R^2 on test set:  0.98


['../src/models/missing_tenth_best_time_estimator_ride.pkl']

In [10]:
X_run_train, X_run_test, y_run_train, y_run_test = train_test_split(X_run, y_run, test_size=0.2, random_state=42)

model_run = LinearRegression()
model_run.fit(X_run_train, y_run_train)
print("Coefficients:", ", ".join(f"{coef:.2f}" for coef in model_run.coef_))
print(f"Intercept: {model_run.intercept_:.2f}")

print(f"Run Model R^2 on test set: {model_run.score(X_run_test, y_run_test): .2f}")

# save model
import joblib
joblib.dump(model_run, '../src/models/missing_tenth_best_time_estimator_run.pkl')

Coefficients: 1.70, -0.69, 2.76, -16.38
Intercept: 67.67
Run Model R^2 on test set:  1.00


['../src/models/missing_tenth_best_time_estimator_run.pkl']